In [82]:
import numpy as np
import itertools
import math
from collections import defaultdict
from itertools import islice

In [83]:
def top_k_kendall_tau(a: list, b: list, k: int, p: float) -> int:
    dist = 0
    p_dist = 0
    a_dict = {a[i]: i for i in range(k)}
    b_dict = {b[i]: i for i in range(k)}
    remaining_b = set(b_dict.keys()) - set(a_dict.keys())

    for i in range(k):
        for j in range(i + 1, k):
            if a[i] not in b_dict and a[j] not in b_dict:
                p_dist += p
            else:
                if a[i] not in b_dict:
                    b_later = True
                elif a[j] not in b_dict:
                    b_later = False
                else:
                    b_later = b_dict[a[i]] > b_dict[a[j]]
                a_later = (i > j)
                dist += a_later != b_later
        for item in remaining_b:
            if a[i] not in b_dict:
                b_later = True
            else:
                b_later = b_dict[a[i]] > b_dict[item]
            a_later = False
            dist += a_later != b_later
    
    p_dist += math.comb(len(remaining_b), 2) * p
    return dist + p_dist

def mallows_prob(center: list, ref_ranking: list, k: int, p: float, beta: float):
    distance = top_k_kendall_tau(center, ref_ranking, k, p)
    return np.exp(-beta * distance)

In [125]:
k = 3
arr_a = [1, 2, 3, 4, 5]
p = 0.5123
beta_a = 0.5

top_k_sets = itertools.permutations(arr_a, k)
count = 0
total_prob = 0
sets = []
probs = []
comb_probs_dict = defaultdict(lambda: 0)

for S in top_k_sets:
    prob_num = mallows_prob(arr_a, S, k, p, beta_a)
    sets.append(S)
    probs.append(prob_num)
    total_prob += prob_num
    count += 1

print("Total:", len(sets))

for i in range(len(sets)):
    prob = probs[i] / total_prob
    print(f"{i}) Ordered Set: {sets[i]}, prob: {round(prob * 100, 3)} %")
    comb_probs_dict[frozenset(sets[i])] += prob

Total: 60
0) Ordered Set: (1, 2, 3), prob: 8.771 %
1) Ordered Set: (1, 2, 4), prob: 5.32 %
2) Ordered Set: (1, 2, 5), prob: 5.32 %
3) Ordered Set: (1, 3, 2), prob: 5.32 %
4) Ordered Set: (1, 3, 4), prob: 3.227 %
5) Ordered Set: (1, 3, 5), prob: 3.227 %
6) Ordered Set: (1, 4, 2), prob: 3.227 %
7) Ordered Set: (1, 4, 3), prob: 1.957 %
8) Ordered Set: (1, 4, 5), prob: 0.711 %
9) Ordered Set: (1, 5, 2), prob: 3.227 %
10) Ordered Set: (1, 5, 3), prob: 1.957 %
11) Ordered Set: (1, 5, 4), prob: 0.711 %
12) Ordered Set: (2, 1, 3), prob: 5.32 %
13) Ordered Set: (2, 1, 4), prob: 3.227 %
14) Ordered Set: (2, 1, 5), prob: 3.227 %
15) Ordered Set: (2, 3, 1), prob: 3.227 %
16) Ordered Set: (2, 3, 4), prob: 1.957 %
17) Ordered Set: (2, 3, 5), prob: 1.957 %
18) Ordered Set: (2, 4, 1), prob: 1.957 %
19) Ordered Set: (2, 4, 3), prob: 1.187 %
20) Ordered Set: (2, 4, 5), prob: 0.431 %
21) Ordered Set: (2, 5, 1), prob: 1.957 %
22) Ordered Set: (2, 5, 3), prob: 1.187 %
23) Ordered Set: (2, 5, 4), prob: 0.43

In [85]:
print("Total:", len(comb_probs_dict))

for comb in comb_probs_dict:
    print(f"Combination: {comb}, prob: {round(comb_probs_dict[comb] * 100, 3)} %")

Total: 10
Combination: frozenset({1, 2, 3}), prob: 27.821 %
Combination: frozenset({1, 2, 4}), prob: 16.874 %
Combination: frozenset({1, 2, 5}), prob: 16.874 %
Combination: frozenset({1, 3, 4}), prob: 10.235 %
Combination: frozenset({1, 3, 5}), prob: 10.235 %
Combination: frozenset({1, 4, 5}), prob: 2.808 %
Combination: frozenset({2, 3, 4}), prob: 6.208 %
Combination: frozenset({2, 3, 5}), prob: 6.208 %
Combination: frozenset({2, 4, 5}), prob: 1.703 %
Combination: frozenset({3, 4, 5}), prob: 1.033 %


In [86]:
stuff = np.arange(24).reshape((2, 3, 4))
stuff

array([[[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]],

       [[12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23]]])

In [87]:
stuff[:2, :3, 2].sum()

np.int64(72)

In [88]:
np.arange(0, 3).sum(-1)

np.int64(3)

In [89]:
np.arange(1, 0 + 1)

array([], dtype=int64)

In [90]:
0.6224593312018546
0.6224593312018546

0.6224593312018546

In [91]:
np.arange(0, 1)

array([0])

In [92]:
"""
def PRIM_POS(S: set, j: np.integer, pos: int, beta: float):
    print("for position:", j, "with size:", pos)
    for i in range(0, pos):
        print("Index:", i, ", Prob:", np.exp(-beta * i))
    print()
    return np.exp(-beta * j) / np.exp(-beta * np.arange(0, pos)).sum(-1)

def PRIM_POS_SEQ(S: set, j: int, pos: int, beta: float, before: bool):
    if j == -1 and before:
        return 0

    prob_before = PRIM_POS(S, np.arange(0, j + 1), pos, beta).sum(0)
    if before:
        return prob_before
    return 1 - prob_before

S = frozenset({1, 2, 3})
j = 1
pos = 3
beta = 0.5

#PRIM_POS(S, j, pos, beta)
PRIM_POS_SEQ(S, j, pos, beta, before=False)
#"""

'\ndef PRIM_POS(S: set, j: np.integer, pos: int, beta: float):\n    print("for position:", j, "with size:", pos)\n    for i in range(0, pos):\n        print("Index:", i, ", Prob:", np.exp(-beta * i))\n    print()\n    return np.exp(-beta * j) / np.exp(-beta * np.arange(0, pos)).sum(-1)\n\ndef PRIM_POS_SEQ(S: set, j: int, pos: int, beta: float, before: bool):\n    if j == -1 and before:\n        return 0\n\n    prob_before = PRIM_POS(S, np.arange(0, j + 1), pos, beta).sum(0)\n    if before:\n        return prob_before\n    return 1 - prob_before\n\nS = frozenset({1, 2, 3})\nj = 1\npos = 3\nbeta = 0.5\n\n#PRIM_POS(S, j, pos, beta)\nPRIM_POS_SEQ(S, j, pos, beta, before=False)\n#'

In [117]:
def PRIM_POS(S: set, j: np.integer, item_count: int, beta: float):
    return np.exp(-beta * j) / np.exp(-beta * np.arange(1, item_count + 1)).sum(-1)

def PRIM_POS_SEQ(S: set, j: int, item_count: int, beta: float, before: bool):
    if j == 0 and before:
        return 0

    prob_before = PRIM_POS(S, np.arange(1, j + 1), item_count, beta).sum(0)
    if before:
        return prob_before
    return 1 - prob_before

def choice_probs(center: list, ref_ranking: frozenset, k: int, beta: float, null_val: int = 0):
    n = len(center)
    center_set = set(center[:k])
    S = center_set.intersection(ref_ranking)
    A_null = ref_ranking.union({null_val})
    A_bar = [a for a in ref_ranking if a not in center_set]
    L = [null_val] + A_bar + center[:k]
    
    r = len(A_bar)
    ell = len(S)
    m = len(L)

    # Reversing
    cur_arr = [(center[i], r + 1 + i) for i in reversed(range(n)) if center[i] in S]
    print("Cur Arr:", cur_arr)
    DP_table = np.zeros((m, k + 1, ell + 1))
    print(r, ell, m)
    print("Table shape:", DP_table.shape)

    for j in range(1, k - ell + 1):
        jind = j - 1
        sampled_at_j_prob = 1/(n - k - j + 1)
        none_sampled_prob = 1
        for jp in range(1, j):
            none_sampled_prob *= (1 - (r / (n - k - jp)))
        DP_table[:r+1, jind, 0] = sampled_at_j_prob * none_sampled_prob
        print("A", sampled_at_j_prob, none_sampled_prob)
        print("B", DP_table[:r+1, jind, 0])
    DP_table[:r+1, k, 0] = (math.comb(n - k - (r + 1), k - ell) / math.comb(n - k, k - ell)) * (1/(r + 1))
    print("one:", (math.comb(n - k - (r + 1), k - ell) / math.comb(n - k, k - ell)) * (1/(r + 1)))
    print("Two:", DP_table[:r+1, k, 0])
    
    print(DP_table)

    #"""
    for q in range(1, ell + 1):
        a_cur, cur_ind = cur_arr[q - 1]
        max_pos = k - ell + q
        print("Curs:", a_cur, cur_ind, q)
        if a_cur not in A_null:
            print("First case")
            DP_table[cur_ind, :, q] = 0
            for j in range(1, k - ell + q + 1):
                jind = j - 1
                DP_table[:, jind, q] = DP_table[:, jind, q-1] * PRIM_POS_SEQ(S, j, max_pos, beta, before=False) + DP_table[:, jind-1, q-1] * PRIM_POS_SEQ(S, j-1, max_pos, beta, before=True)
            DP_table[:, k, q] = DP_table[:, k, q-1]
        else:
            print("Second case")
            print("looping")
            for j in range(1, k - ell + q + 1):
                jind = j - 1
                print(j, DP_table[:, jind, q-1], PRIM_POS_SEQ(S, j, max_pos, beta, before=False))
                DP_table[:, jind, q] = DP_table[:, jind, q-1] * PRIM_POS_SEQ(S, j, max_pos, beta, before=False)
                print(PRIM_POS(S, j, max_pos, beta), DP_table[:, j:, q-1].sum((0, 1)))
                DP_table[cur_ind, jind, q] = PRIM_POS(S, j, max_pos, beta) * DP_table[:, jind:, q-1].sum((0, 1))
                print("and a")
            print("end loop")
        print("Again")
        print(DP_table)
    #"""
    
    print("Total probs:", DP_table.sum((0, 1)))
    item_probs = DP_table[:, :k, ell].sum((1))
    return item_probs

In [145]:
arr_h = [1, 2, 3, 4, 5]
beta_h = 0.5

ranking_a = next(islice(comb_probs_dict.keys(), 0, None))
print(ranking_a)

choice_probs(arr_h, ranking_a, k, beta_h)

frozenset({1, 2, 3})
Cur Arr: [(3, 3), (2, 2), (1, 1)]
0 3 4
Table shape: (4, 4, 4)
one: 1.0
Two: [1.]
[[[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [1. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]]
Curs: 3 3 1
Second case
looping
1 [0. 0. 0. 0.] 0.0
1.0 1.0
and a
end loop
Again
[[[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [1. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]

 [[0. 1. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]
  [0. 0. 0. 0.]]]
Curs: 2 2 2
Second case
looping
1 [0. 0. 0. 1.] 0.3775406687981454
0.6224593312018546 0.0
and a
2 [0. 0. 0. 0.] 0.0
0.37754066879814546 0.0
and a
end loop
Again
[[[0.         0.         0.         0.        ]
  [0.         0.         0.         0.        ]
  [0.         0.         0.    

array([0.        , 0.50648039, 0.30719589, 0.18632372])

In [146]:
ranking_a = next(islice(comb_probs_dict.keys(), 6, None))
print(ranking_a)

choice_probs(arr_h, ranking_a, k, beta_h)

frozenset({2, 3, 4})
Cur Arr: [(3, 4), (2, 3)]
1 2 5
Table shape: (5, 4, 3)
A 0.5 1
B [0.5 0.5]
one: 0.0
Two: [0. 0.]
[[[0.5 0.  0. ]
  [0.  0.  0. ]
  [0.  0.  0. ]
  [0.  0.  0. ]]

 [[0.5 0.  0. ]
  [0.  0.  0. ]
  [0.  0.  0. ]
  [0.  0.  0. ]]

 [[0.  0.  0. ]
  [0.  0.  0. ]
  [0.  0.  0. ]
  [0.  0.  0. ]]

 [[0.  0.  0. ]
  [0.  0.  0. ]
  [0.  0.  0. ]
  [0.  0.  0. ]]

 [[0.  0.  0. ]
  [0.  0.  0. ]
  [0.  0.  0. ]
  [0.  0.  0. ]]]
Curs: 3 4 1
Second case
looping
1 [0.5 0.5 0.  0.  0. ] 0.3775406687981454
0.6224593312018546 0.0
and a
2 [0. 0. 0. 0. 0.] 0.0
0.37754066879814546 0.0
and a
end loop
Again
[[[0.5        0.18877033 0.        ]
  [0.         0.         0.        ]
  [0.         0.         0.        ]
  [0.         0.         0.        ]]

 [[0.5        0.18877033 0.        ]
  [0.         0.         0.        ]
  [0.         0.         0.        ]
  [0.         0.         0.        ]]

 [[0.         0.         0.        ]
  [0.         0.         0.        ]
  [0. 

array([0.09316186, 0.09316186, 0.        , 0.50648039, 0.30719589])